# A.3 文件多了怎麼找相關段落？

# A：上下文學習與 RAG

前置：第7章prompt與對話。不需要圖片或MoE。把模型想成可以讀你遞給它的小抄的答題者：先手動給例子，再從本地文件找相關段落。檢索成功不等於回答一定正確。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像先找小抄，再核對答案來源**

找到對文件與用對資料，是兩個不同分數。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rag.svg")))

**先想一想：**問題同義詞不同，詞匹配可能錯過嗎？

文件多時先找可能相關段落。最小版本用字詞交集排序，像索引卡找關鍵詞；它便宜透明，但不等於語義檢索。

**把直覺寫成數學：**score=Σ共享詞頻；本工具中文按字、英文按單詞，方便從零理解。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.retrieval import retrieve

docs = [{"id": "d1", "text": "貓喜歡曬太陽"}, {"id": "d2", "text": "書店地址是青街8號"}]
hits = retrieve("書店地址", docs, k=1)
print(hits)
assert hits[0]["id"] == "d2"

**如何讀結果：**查到正確文件後還需抽取答案；詞交集可被重複關鍵詞或幹擾內容騙到。

**只改一件事：**只把查詢換成「營業處在哪」，檢查此簡單檢索的侷限。
